# Multi-Cluster Time Pattern Extraction

This notebook improves upon the original dominant-bucket algorithm.

The previous implementation could only discover a single routine per
(device, action) pair.

Example:

living_room_light ON

04:00 every morning
18:00 every evening

The old algorithm would keep only one of these routines.

The new algorithm discovers multiple significant routine clusters while
filtering out noise.

In [33]:
from collections import defaultdict
from datetime import datetime
import statistics
import math

In [34]:
# Example dataset

events = [
    
    # midhnight routine
    ("living_room_light", "ON", "2026-06-01 23:58"),
    ("living_room_light", "ON", "2026-06-02 00:01"),
    ("living_room_light", "ON", "2026-06-03 23:59"),
    ("living_room_light", "ON", "2026-06-04 00:02"),
    ("living_room_light", "ON", "2026-06-05 23:57"),
    ("living_room_light", "ON", "2026-06-06 00:03"),
    ("living_room_light", "ON", "2026-06-07 23:58"),
    ("living_room_light", "ON", "2026-06-08 00:01"),
    ("living_room_light", "ON", "2026-06-09 23:59"),
    
    # Morning routine
    ("living_room_light", "ON", "2026-06-01 04:01"),
    ("living_room_light", "ON", "2026-06-02 04:03"),
    ("living_room_light", "ON", "2026-06-03 04:00"),
    ("living_room_light", "ON", "2026-06-04 04:05"),
    ("living_room_light", "ON", "2026-06-05 04:02"),
    ("living_room_light", "ON", "2026-06-06 04:01"),
    ("living_room_light", "ON", "2026-06-07 04:04"),

    # Evening routine
    ("living_room_light", "ON", "2026-06-01 18:00"),
    ("living_room_light", "ON", "2026-06-02 18:02"),
    ("living_room_light", "ON", "2026-06-03 18:01"),
    ("living_room_light", "ON", "2026-06-04 18:03"),
    ("living_room_light", "ON", "2026-06-05 18:04"),
    ("living_room_light", "ON", "2026-06-06 18:02"),
    ("living_room_light", "ON", "2026-06-07 18:01"),
    ("living_room_light", "ON", "2026-06-08 18:03"),
    ("living_room_light", "ON", "2026-06-09 18:02"),
    ("living_room_light", "ON", "2026-06-10 18:00"),

    # Noise
    ("living_room_light", "ON", "2026-06-02 11:15"),
    ("living_room_light", "ON", "2026-06-11 22:30"),
]

In [35]:
# convert to date and time

parsed_events = []

for device, action, ts in events:
    parsed_events.append(
        {
            "device": device,
            "action": action,
            "timestamp": datetime.strptime(
                ts,
                "%Y-%m-%d %H:%M"
            )
        }
    )

## Step 1

Convert timestamps into minutes since midnight.

Examples:

04:00 -> 240

18:00 -> 1080

This allows numerical clustering.

- Added circular distance and circular mean functions

In [36]:
# helper functions

def minutes_of_day(ts):
    return ts.hour * 60 + ts.minute


def fmt_hhmm(minutes):
    m = int(round(minutes)) % 1440
    return f"{m//60:02d}:{m%60:02d}"


def circular_distance(a, b):
    """
    Smallest distance between two times on a 24-hour clock.
    """

    diff = abs(a - b)

    return min(
        diff,
        1440 - diff
    )
    
    
def circular_mean(minutes):

    angles = [
        2 * math.pi * m / 1440
        for m in minutes
    ]

    x = sum(
        math.cos(a)
        for a in angles
    )

    y = sum(
        math.sin(a)
        for a in angles
    )

    mean_angle = math.atan2(y, x)

    if mean_angle < 0:
        mean_angle += 2 * math.pi

    return (
        mean_angle
        * 1440
        / (2 * math.pi)
    )
    
    
def circular_stddev(minutes):

    center = circular_mean(minutes)

    distances = [
        circular_distance(
            m,
            center
        )
        for m in minutes
    ]

    return statistics.pstdev(distances)

## Step 2

Pattern extraction is performed independently for every:

(device, action)

pair.

This prevents unrelated device behaviours from influencing each other.

In [37]:
groups = defaultdict(list)

for e in parsed_events:

    key = (
        e["device"],
        e["action"]
    )

    groups[key].append(
        minutes_of_day(
            e["timestamp"]
        )
    )

groups

defaultdict(list,
            {('living_room_light', 'ON'): [1438,
              1,
              1439,
              2,
              1437,
              3,
              1438,
              1,
              1439,
              241,
              243,
              240,
              245,
              242,
              241,
              244,
              1080,
              1082,
              1081,
              1083,
              1084,
              1082,
              1081,
              1083,
              1082,
              1080,
              675,
              1350]})

In [38]:
#Create Buckets

BUCKET_SIZE = 30

minutes = groups[
    ("living_room_light", "ON")
]

bucket_counts = defaultdict(list)

for m in minutes:

    bucket_id = m // BUCKET_SIZE

    bucket_counts[bucket_id].append(m)

bucket_counts

defaultdict(list,
            {47: [1438, 1439, 1437, 1438, 1439],
             0: [1, 2, 3, 1],
             8: [241, 243, 240, 245, 242, 241, 244],
             36: [1080, 1082, 1081, 1083, 1084, 1082, 1081, 1083, 1082, 1080],
             22: [675],
             45: [1350]})

## Step 3

Unlike the previous algorithm, we will not select only the dominant bucket.

Instead, every sufficiently populated bucket becomes a candidate routine.

In [39]:
# Display histogram

for bucket_id in sorted(bucket_counts):

    print(
        bucket_id,
        len(bucket_counts[bucket_id])
    )

0 4
8 7
22 1
36 10
45 1
47 5


Buckets with very little support are likely random events.

We discard them immediately.

In [40]:
MIN_OCCURRENCES = 3

candidate_buckets = []

for bucket_id, values in bucket_counts.items():

    if len(values) >= MIN_OCCURRENCES:
        candidate_buckets.append(bucket_id)

candidate_buckets

[47, 0, 8, 36]

## Step 4

Neighbouring buckets often belong to the same routine.

Example:

18:50
19:00
19:10

These should become one cluster rather than three patterns.

In [41]:
candidate_buckets.sort()

merged_clusters = []

current = [candidate_buckets[0]]

for bucket in candidate_buckets[1:]:

    if bucket == current[-1] + 1:

        current.append(bucket)

    else:

        merged_clusters.append(current)

        current = [bucket]

merged_clusters.append(current)

merged_clusters

[[0], [8], [36], [47]]

In [42]:
TOTAL_BUCKETS = 1440 // BUCKET_SIZE

if (
    len(merged_clusters) > 1
    and merged_clusters[0][0] == 0
    and merged_clusters[-1][-1] == TOTAL_BUCKETS - 1
):
    merged_clusters[0] = (
        merged_clusters[-1]
        + merged_clusters[0]
    )

    merged_clusters.pop()

In [43]:
# Recover events inside each cluster

time_clusters = []

for cluster in merged_clusters:

    values = []

    for bucket in cluster:

        values.extend(
            bucket_counts[bucket]
        )

    time_clusters.append(values)

time_clusters

[[1438, 1439, 1437, 1438, 1439, 1, 2, 3, 1],
 [241, 243, 240, 245, 242, 241, 244],
 [1080, 1082, 1081, 1083, 1084, 1082, 1081, 1083, 1082, 1080]]

## Step 5

Each cluster now represents an independent routine candidate.

We calculate:

- Mean time
- Standard deviation
- Support count

In [44]:
# Calculate statistics

cluster_stats = []

for cluster in time_clusters:

    mean_time = circular_mean(cluster)

    stddev = (
        circular_stddev(cluster)
        if len(cluster) > 1
        else 0
    )

    cluster_stats.append(
        {
            "mean": mean_time,
            "stddev": stddev,
            "occurrences": len(cluster)
        }
    )

cluster_stats

[{'mean': 1439.7777726763336, 'stddev': 0.8093732476585174, 'occurrences': 9},
 {'mean': 242.28571012268768, 'stddev': 0.7851188298580953, 'occurrences': 7},
 {'mean': 1081.7999995430725, 'stddev': 0.6916645212228101, 'occurrences': 10}]

In [45]:
# Relative Dominance Threshold

dominant_support = max(
    c["occurrences"]
    for c in cluster_stats
)

dominant_support

10

## Step 6

A cluster should not survive merely because it passes the minimum occurrence threshold.

We also require it to represent a meaningful fraction of the strongest routine.

This prevents tiny accidental clusters from becoming patterns.

In [46]:
RELATIVE_THRESHOLD = 0.40

significant_clusters = []

for cluster in cluster_stats:

    if (
        cluster["occurrences"]
        >= dominant_support *
        RELATIVE_THRESHOLD
    ):
        significant_clusters.append(cluster)

significant_clusters

[{'mean': 1439.7777726763336, 'stddev': 0.8093732476585174, 'occurrences': 9},
 {'mean': 242.28571012268768, 'stddev': 0.7851188298580953, 'occurrences': 7},
 {'mean': 1081.7999995430725, 'stddev': 0.6916645212228101, 'occurrences': 10}]

In [47]:
# Confidence Scoring

def support_score(count):

    return min(
        count / 10,
        1.0
    )


def consistency_score(
    stddev,
    tolerance=30
):

    return max(
        0,
        1 - stddev / tolerance
    )

In [48]:
# Final Patterns

patterns = []

for cluster in significant_clusters:

    support = support_score(
        cluster["occurrences"]
    )

    consistency = consistency_score(
        cluster["stddev"]
    )

    confidence = (
        support +
        consistency
    ) / 2

    patterns.append(
        {
            "usual_time":
            fmt_hhmm(cluster["mean"]),

            "occurrences":
            cluster["occurrences"],

            "confidence":
            round(confidence,3)
        }
    )

patterns

[{'usual_time': '00:00', 'occurrences': 9, 'confidence': 0.937},
 {'usual_time': '04:02', 'occurrences': 7, 'confidence': 0.837},
 {'usual_time': '18:02', 'occurrences': 10, 'confidence': 0.988}]

# Results

The original algorithm would have produced:

18:02

only.

The multi-cluster algorithm successfully discovers:

04:02
18:02

while discarding:

11:15
22:30

as noise.

Advantages:

✓ Multiple daily routines

✓ Deterministic

✓ Explainable

✓ Minimal changes to existing code

✓ Compatible with existing confidence scoring

Future improvements:

- Circular time handling (midnight wraparound)
- DBSCAN clustering
- Seasonal patterns
- Day-of-week specific routines